## Setup

In [ ]:
import sys
import os
import importlib

from tabulate import tabulate
import matplotlib.pyplot as plt

sys.path.append("../src")

import utils
import plot
import stats

# Reload modules to apply any changes
importlib.reload(utils)
importlib.reload(plot)
importlib.reload(stats)

In [ ]:
FILENAME = os.getenv("FILENAME", "UFABC_PLT_combined")
METRIC = os.getenv("METRIC", "entropy")
DST = f"../results/{FILENAME}/"
df = utils.load(f"../results/{FILENAME}/metrics.csv")

print(f"{FILENAME=}")
print(f"{METRIC=}")
print(f"{df.shape=}")

In [ ]:
grouped = df.groupby(["id", "category", "concept"], as_index=False)
dfx = grouped.mean(numeric_only=True)
print(tabulate(dfx.head(3), headers="keys", showindex=False))

## Analyze (all metrics)

In [ ]:
importlib.reload(stats)

metrics = [
    "distance_centroid_order",
    "distance_next",
    "distance_derivative",
    "distance_second_derivative",
    "distance_centroid_static",
    "vel_magnitude",
    "acc_magnitude",
]

for i, metric in enumerate(metrics):
    print(f"\n[{i + 1}/{len(metrics)}] Analyzing '{metric}'")
    summary = stats.mean_sd_by_group(dfx, "concept", metric)
    summary.to_csv(f"{DST}/mean-sd-by-{metric}.csv", index=False)

    plot.boxplot(dfx, "concept", metric)
    plt.savefig(f"{DST}/boxplot-{metric}.png", bbox_inches="tight")
    plt.clf()

    res, pred, pairs = plot.boxplot_with_model(dfx, metric)
    plt.savefig(f"{DST}/boxplot-with-model-{metric}.png", bbox_inches="tight")
    plt.clf()

    pairs.to_csv(f"{DST}/tukey-pairwise-{metric}.csv", index=False)

## Analyze (single metric)

In [ ]:
summary = stats.mean_sd_by_group(dfx, "concept", METRIC)
summary.to_csv(f"{DST}/mean-sd-by-{METRIC}.csv", index=False)
print(tabulate(summary, headers="keys", showindex=False))

In [ ]:
plot.boxplot(dfx, "concept", METRIC)
plt.show()

In [ ]:
res, pred, pairs = plot.boxplot_with_model(dfx, METRIC, verbose=True)
plt.show()

In [ ]:
print(tabulate(pairs, headers="keys", showindex=False))

In [ ]:
print(res.summary())